# Attendance rates and Precision/Recall/F1 with infer-ci (INF-43)

Accuracy, error rates and Precision/Recall/F1 of the attendance system, with 95% Wilson confidence intervals.

In [ ]:
import pandas as pd
from infer_ci import accuracy_score, precision_score, recall_score, f1_score

TOLERANCE_MIN = 5  # |system - real| <= 5 minutes is a match (attendance-data-guide.pdf)

df = pd.read_csv("attendance_times.csv", dtype={"person_id": str},
                 parse_dates=["real_in", "system_in", "real_out", "system_out"])
print(df.head())

## 1. One row per person, day and direction

In [ ]:
id_cols = [c for c in df.columns if c not in ["real_in","system_in","real_out","system_out"]]
rows = []
for d, r, s in [("IN","real_in","system_in"), ("OUT","real_out","system_out")]:
    t = df[id_cols].copy()
    t["direction"] = d
    t["real"] = df[r]
    t["system"] = df[s]
    rows.append(t)
long = pd.concat(rows, ignore_index=True)
long.head()

## 2. Result column (rule from the PDF)

In [ ]:
def classify(real, system):
    if pd.isna(real) and pd.isna(system):
        return "absent"
    if pd.isna(system):
        return "missed"
    if pd.isna(real):
        return "extra"
    diff = (system - real).total_seconds() / 60
    if abs(diff) <= TOLERANCE_MIN:
        return "match"
    return "late" if diff > 0 else "early"

long["result"] = [classify(a, b) for a, b in zip(long["real"], long["system"])]
long = long[long["result"] != "absent"].reset_index(drop=True)
print(long["result"].value_counts())

## 3. Rates table (accuracy and error rates)

In [ ]:
rates = long[["direction", "result"]].copy()
rates["y_true"] = 1
rates["is_match"] = (rates.result == "match").astype(int)
rates["is_late_early"] = rates.result.isin(["late", "early"]).astype(int)
rates["is_missed"] = (rates.result == "missed").astype(int)
rates["is_extra"] = (rates.result == "extra").astype(int)
print(len(rates), rates.direction.value_counts().to_dict())

## 4. Events table (Precision/Recall/F1)

Accuracy counts a late or early record as one wrong pass, but for F1 it is both a missed real pass (FN) and a wrong system record (FP), so the events table has two rows for it.

In [ ]:
MAP = {"match": [(1, 1)], "missed": [(1, 0)], "extra": [(0, 1)], "late": [(1, 0), (0, 1)], "early": [(1, 0), (0, 1)]}
ev = [(d, yt, yp) for d, res in zip(long.direction, long.result) for yt, yp in MAP[res]]
events = pd.DataFrame(ev, columns=["direction", "y_true", "y_pred"])
tp = ((events.y_true == 1) & (events.y_pred == 1)).sum()
fp = ((events.y_true == 0) & (events.y_pred == 1)).sum()
fn = ((events.y_true == 1) & (events.y_pred == 0)).sum()
print("TP/FP/FN:", tp, fp, fn, "rows:", len(events))

## 5–6. Metrics for all, IN and OUT

In [ ]:
def metrics(r, e):
    out = {}
    out["Accuracy"] = accuracy_score(r.y_true, r.is_match, method="wilson")
    out["Late/early rate"] = accuracy_score(r.y_true, r.is_late_early, method="wilson")
    out["Missed rate"] = accuracy_score(r.y_true, r.is_missed, method="wilson")
    out["Extra rate"] = accuracy_score(r.y_true, r.is_extra, method="wilson")
    out["Precision"] = precision_score(e.y_true, e.y_pred, average="binary", method="wilson")
    out["Recall"] = recall_score(e.y_true, e.y_pred, average="binary", method="wilson")
    out["F1"] = f1_score(e.y_true, e.y_pred, average="binary", method="wilson")
    return out

res = []
for g in ["all", "IN", "OUT"]:
    r = rates if g == "all" else rates[rates.direction == g]
    e = events if g == "all" else events[events.direction == g]
    for m, (v, (lo, hi)) in metrics(r, e).items():
        res.append({"group": g, "metric": m, "value": float(v), "lower": float(lo), "upper": float(hi)})
results = pd.DataFrame(res).round(4)
results

## Check

In [ ]:
assert abs(rates.is_match.mean() - results.query("group=='all' and metric=='Accuracy'").value.iloc[0]) < 1e-3
print("accuracy by hand:", round(rates.is_match.mean(), 4))
print(results.to_string(index=False))